In [1]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)
from jax_sn.operators.transport_operator import TransportOperatorBatched, TransportOperatorMultipleAnglesVmap, TransportOperatorVmap, TransportOperatorBatchedPallas, TransportOperatorMultipleAngles
import equinox as eqx
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)
from finite_elements.test_moment_scatter_operator import _setup_all_sb

In [12]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain
from jax_sn.source import Source
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
import lineax as lx

In [88]:
from finite_elements.test_breeding_blanket import bb_data_dict, base_region_sizes, base_resolutions,  _get_bb_args, _setup_bb_args, _get_bb_args


def setup_mgop(dim, degree):
    MGOP, source_eg0, solution_domain = _setup_bb_args(dim, degree, base_region_sizes(), base_resolutions(), 5.0, 10, 3)
    transport_operator                = TransportOperatorMultipleAnglesVmap(TransportOperatorBatched(MGOP.transport_operator.solution_domain_data, MGOP.transport_operator.topological_sorted_angle, 0, 100))
    MGOP                              = eqx.tree_at(lambda t : t.transport_operator, MGOP, transport_operator)
    return MGOP, source_eg0, solution_domain

def setup_mgop_mod_theta(dim, degree, xs_data_indices, dx):
    MGOP, source_eg0, solution_domain = setup_mgop(dim, degree)    

    total_xs = MGOP.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data.total
    scattering_xs = MGOP.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data.scattering    
    if xs_data_indices[0] == 'total':
        total_xs = total_xs.at[xs_data_indices[1]].add(dx)
        
    elif xs_data_indices[0] == 'scattering':
        scattering_xs = scattering_xs.at[xs_data_indices[1]].add(dx)
    else:
        raise ValueError("Invalid xs_data_indices")    
    
    sgop = MGOP.to_single_group_operator(0)

    xs_data_new = CrossSectionData(total_xs, scattering_xs)
    sgop_new = eqx.tree_at(lambda sgop : [sgop.transport_operator.solution_domain_data.domain.cross_sections_indexed.cross_section_data, sgop.inscatter_operator.cross_sections_indexed.cross_section_data], sgop,[xs_data_new,xs_data_new])
    return sgop_new, source_eg0, solution_domain



MGOP, source_eg0, solution_domain = setup_mgop(2,2)

In [89]:
def total_solve(sgop, source_eg0, solver):
    rhs = sgop.moment_operator.mv(sgop.transport_operator.inv(sgop.mass_operator.mv(source_eg0)))
    return lx.linear_solve(sgop, rhs, solver)

def solve_mgop_mod_theta(dim,degree, xs_data_indices, dx, solver):
    sgop_new, source_eg0, solution_domain = setup_mgop_mod_theta(dim, degree, xs_data_indices, dx)
    return total_solve(sgop_new, source_eg0, solver).value.sum()



In [97]:
from jax_sn.tools.memory_analysis import OperatorAnalysis
from jax_sn.tools.memory_analysis import TotalOperatorSigmaAllParametrised
bicgstab = lx.BiCGStab(1e-10, 1e-10, max_steps=100)

In [98]:

def dx_mgop_mod_theta(dim, degree, xs_data_indices, dx, solver):
    return (solve_mgop_mod_theta(dim,degree, xs_data_indices, dx, solver) - solve_mgop_mod_theta(dim,degree, xs_data_indices, 0.0, solver)) / dx


dx_mgop_mod_theta(2,2, ['total',(1,0)], 1e-5, bicgstab)

Array(-8.40590556, dtype=float64)

In [ ]:
def sum_f(sgop, rhs, theta, solver):        
    sgop_new = TotalOperatorSigmaAllParametrised.reconstruct_operator(sgop, theta)
    return total_solve(sgop_new, rhs, solver).value.sum()  

In [109]:

sgop = MGOP.to_single_group_operator(0)


sum_grad_f = eqx.filter_jit(jax.grad(sum_f, argnums = (2,)))

all_derivatives = sum_grad_f(sgop, source_eg0, TotalOperatorSigmaAllParametrised.theta_base(sgop), bicgstab)[0]

In [110]:
def index_all_derivatives(xs_data_indices, all_derivatives):
    if xs_data_indices[0] == 'total':
        return all_derivatives.total[xs_data_indices[1]]
    elif xs_data_indices[0] == 'scattering':
        return all_derivatives.scattering[xs_data_indices[1]]
    else:
        raise ValueError("Invalid xs_data_indices"
                         )
dx = 1e-5

test_total_0 = ('total',(2,0))

print("Testing finite difference vs. automatic differentiation for total xs at index (2,0)")
print("Automatic differentiation:", index_all_derivatives(test_total_0, all_derivatives))
print("Finite difference:        ", dx_mgop_mod_theta(2,2, test_total_0, dx, bicgstab) )

Testing finite difference vs. automatic differentiation for total xs at index (2,0)
Automatic differentiation: -56.30918713881783
Finite difference:         -56.30914586163271


In [112]:
    
test_scattering = ('scattering',(2,0,0,0))

print("Testing finite difference vs. automatic differentiation for scattering xs at index (2,0,0)")
print("Automatic differentiation:", index_all_derivatives  (test_scattering, all_derivatives))
print("Finite difference:        ", dx_mgop_mod_theta(2,2, test_scattering, dx, bicgstab) )

Testing finite difference vs. automatic differentiation for scattering xs at index (2,0,0)
Automatic differentiation: 61.90003744664815
Finite difference:         61.90006729411834


In [113]:
test_scattering = ('scattering',(2,1,0,0))
print("Testing finite difference vs. automatic differentiation for scattering xs at index (2,1,0)")
print("Automatic differentiation:", index_all_derivatives  (test_scattering, all_derivatives))
print("Finite difference:        ", dx_mgop_mod_theta(2,2, test_scattering, dx, bicgstab) )

Testing finite difference vs. automatic differentiation for scattering xs at index (2,0,0)
Automatic differentiation: -5.26108963680713
Finite difference:         -5.261090086605691
